# Train YOLO Detect on Colab (T4) — `detectset.ndjson` (SELECTED 6-class, STANDARD overwrite baseline)

**Classes (`SELECTED`, IDs = order):** `cell phone` (0), `laptop` (1), `wallet` (2), `bottle` (3), `bag` (4, merges old `handbag` + `backpack`), `cup` (5). Edit `SELECTED` in the setup cell per run — everything downstream derives from it. Keep identical to the grafted notebook for a fair ablation.

**Base:** `yolo11x.pt` (COCO). STANDARD baseline for the graft ablation: trains straight from the base, letting Ultralytics do its default `nc=80 → nc=6` head reset (fresh random classifier, backbone/neck transferred). Identical data, schedule, and hyperparameters to the grafted notebook — the head init is the only variable. Compare per-class AP50/AP50-95/recall at ~10 epochs and at convergence.

**Dataset:** tagger-curated + manually checked `detectset.ndjson` (Ultralytics Platform NDJSON export, `task: detect`, `class_names` in `SELECTED` order) — upload it to `Drive/Thesis_Workspace/datasets/`. The stage cell validates it and converts to YOLO format (downloads images via record urls, hash-cached on re-runs).

**Colab setup:** `colab.research.google.com` → Upload this notebook → Runtime → Change runtime type → **T4 GPU** → Runtime → Run all (authorise Drive when asked).

**Session loop (free tier disconnects!):** run setup → stage → train (no seed cell here). On disconnect, re-run setup (mount only), then setup → stage (re-convert is cached) → resume instead of train. Always sync weights to Drive (train/resume do it automatically).

**Recipe:** Ultralytics defaults for augmentation (same `_COMMON` as the grafted notebook, ablated variable = head init only). (`mosaic=1.0`, `close_mosaic=10`, `scale=0.5`, `translate=0.1`, `fliplr=0.5`, `hsv 0.015/0.7/0.4`; `copy_paste` removed — proven no-op for box-only detect: `CopyPaste` early-returns on empty segments) + same two-phase schedule (frozen-backbone warmup, then full fine-tune). Deliberate non-defaults, all documented: `SGD + lr0=0.01 + cos_lr` (classic YOLO recipe), `patience=20` (Colab-budget early stopping vs default 100), `freeze=10` (backbone convention shared with `model_train_pose.ipynb`). Low-RAM T4 safe: `batch=16`, `workers=2`, `cache=disk` — record the observed per-epoch `GPU_mem` for the thesis.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q "ultralytics==8.4.127" nest_asyncio
!nvidia-smi -L
!free -h | head -3  # want 25GB+ total; ~13GB runtimes must use the disk-cache settings below

from pathlib import Path
import torch, ultralytics
import nest_asyncio
nest_asyncio.apply()  # lets ultralytics' async ul:// + ndjson loader run inside Jupyter's event loop

# Self-defined run classes. IDs derive from order — never reorder, only append/replace deliberately.
# Canonical COCO naming (tagger rebound phone -> cell phone). wallet = manual-only (no COCO match).
SELECTED = ["cell phone", "laptop", "wallet", "bottle", "bag", "cup"]
BASE = "yolo11x.pt"  # x-arch COCO source (auto-downloads on first run)

PROJECT = Path("/content/drive/MyDrive/Thesis_Workspace")
DRIVE_DATASETS = PROJECT / "datasets"
NDJSON_NAME = "detectset.ndjson"  # tagger export; upload to Drive/Thesis_Workspace/datasets/
NDJSON_DRIVE = DRIVE_DATASETS / NDJSON_NAME
NDJSON_LOCAL = Path("/content/datasets") / NDJSON_NAME
DATA_SRC = None  # set by the stage cell: converted data.yaml derived from the ndjson
RUNS_DIR = Path("/content/runs")
RUN_NAME = "standard_11X_colab"

assert len(set(SELECTED)) == len(SELECTED), f"SELECTED must be unique: {SELECTED}"
print(f"SELECTED ({len(SELECTED)}): {SELECTED}")
print(f"CLASS_MAP: {{name: i for i, name in enumerate(SELECTED)}} = "
      f"{{{', '.join(f'{c}: {i}' for i, c in enumerate(SELECTED))}}}")

assert torch.cuda.is_available(), "No GPU! Runtime -> Change runtime type -> T4."
print(torch.cuda.get_device_name(0))
print(f"ultralytics {ultralytics.__version__} | torch {torch.__version__}")
assert (DRIVE_DATASETS / NDJSON_NAME).exists(), \
    f"Upload {NDJSON_NAME} to Drive/Thesis_Workspace/datasets/ first."
device = 0


In [ ]:
# Stage detectset.ndjson on Colab disk (always re-copy: the export is small text,
# so a fresh tagger export is picked up on re-run), validate it, then convert
# to YOLO format via ultralytics' ndjson loader (same path model_train_pose uses).
import json as _json
import shutil
from collections import Counter

Path("/content/datasets").mkdir(parents=True, exist_ok=True)
RUNS_DIR.mkdir(parents=True, exist_ok=True)
shutil.copy(NDJSON_DRIVE, NDJSON_LOCAL)
print(f"Staged {NDJSON_DRIVE} -> {NDJSON_LOCAL} ({NDJSON_LOCAL.stat().st_size/1e6:.2f} MB)")

# Validate the ndjson itself: dataset header + per-split counts + class histogram.
_lines = [ln for ln in NDJSON_LOCAL.read_text(encoding="utf-8").splitlines() if ln.strip()]
assert _lines, "detectset.ndjson is empty"
_head = _json.loads(_lines[0])
assert _head.get("type") == "dataset", f"first line must be the dataset record, got: {str(_head)[:120]}"
_task = _head.get("task", "detect")
assert _task == "detect", f"detectset.ndjson task={_task!r}, expected 'detect'"
_cn = {int(k): str(v) for k, v in (_head.get("class_names") or {}).items()}
_ordered = [_cn[i] for i in sorted(_cn)] if _cn else None
if _ordered is not None:
    assert _ordered == SELECTED, f"ndjson class_names {_ordered} != SELECTED {SELECTED} (fix tagger export order)"
    print(f"class_names OK ({len(_ordered)}): {_ordered}")
else:
    print("WARNING: no class_names header — falling back to label-id histogram check only.")
_recs = [_json.loads(ln) for ln in _lines[1:]]
_splits = Counter(r.get("split") for r in _recs)
print(f"records: {len(_recs)} | splits={dict(_splits)}")
assert "train" in _splits, f"ndjson missing 'train' split: {dict(_splits)}"
_hist = Counter()
_no_url = 0
for _r in _recs:
    assert _r.get("split") in ("train", "val", "test"), f"bad split: {_r.get('split')!r}"
    assert isinstance(_r.get("file"), str) and _r["file"], "image record missing 'file'"
    if not _r.get("url"):
        _no_url += 1
    for _labels in (_r.get("annotations") or {}).values():
        for _lab in _labels:
            if _lab:
                _hist[int(_lab[0])] += 1
        break  # converter writes the first annotation key only — mirror it here
print(f"class_hist(id: count)={dict(sorted(_hist.items()))} | records without url={_no_url}")
assert set(_hist) <= set(range(len(SELECTED))), \
    f"label ids {sorted(_hist)} outside SELECTED range 0..{len(SELECTED)-1}"
if _no_url and not (NDJSON_LOCAL.parent / "images").is_dir():
    raise FileNotFoundError(
        f"{_no_url} records have no url and no local images/ dir sits next to the ndjson — "
        "either re-export with urls or stage the images folder alongside detectset.ndjson.")

# Convert (downloads images via urls on first run; hash-cached on re-runs) -> data.yaml.
from ultralytics.data.utils import convert_ndjson_to_yolo_if_needed
DATA_SRC = Path(convert_ndjson_to_yolo_if_needed(str(NDJSON_LOCAL))).resolve()
assert DATA_SRC.is_file(), f"conversion produced no data.yaml: {DATA_SRC}"
print(f"DATA_SRC -> {DATA_SRC}")
import yaml as _yaml
print(_yaml.safe_load(DATA_SRC.read_text(encoding="utf-8")))


In [ ]:
# Two-phase train straight from BASE (overwrite baseline). Low-RAM T4 safe: batch=16, workers=2, cache on disk.
# Ultralytics resets the head nc=80 -> nc=6 here (random init); backbone/neck transfer.
# Phase 1: frozen-backbone warmup (fresh head settles, backbone intact).
# Phase 2: full fine-tune from phase-1 weights.
from ultralytics import YOLO
import shutil
from pathlib import Path as _P

FREEZE = 10          # backbone depth (same convention as model_train_pose FREEZE=10)
FREEZE_EPOCHS = 10
EPOCHS = 100

assert DATA_SRC and DATA_SRC.is_file(), "Run the stage cell first (DATA_SRC not set)."
_COMMON = dict(
    data=str(DATA_SRC),
    device=device,
    amp=True,
    imgsz=640,
    batch=16,
    workers=2,
    optimizer="SGD",
    lr0=0.01,
    cos_lr=True,
    mosaic=1.0,
    close_mosaic=10,
    scale=0.5,
    translate=0.1,
    fliplr=0.5,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    cache="disk",
    plots=True,
    save_period=10,
    project=str(RUNS_DIR),
    patience=20,
    verbose=True,
    seed=0,
)

_m1 = YOLO(BASE)  # auto-downloads; nc-override happens at train start
_m1.train(epochs=FREEZE_EPOCHS, freeze=FREEZE, name=RUN_NAME, exist_ok=False, **_COMMON)
_p1_last = _P(_m1.trainer.save_dir) / "weights" / "last.pt"
assert _p1_last.exists(), f"phase-1 last.pt missing: {_p1_last}"
print(f"Phase-1 (frozen warmup) done -> {_p1_last}")

_m2 = YOLO(str(_p1_last))
_m2.train(epochs=EPOCHS, name=RUN_NAME + "-full", exist_ok=False, **_COMMON)

# Colab disks are ephemeral: sync EVERY matching run dir (warmup + full + bumps).
_synced = []
for _run in sorted(RUNS_DIR.glob(RUN_NAME + "*")):
    if not (_run.is_dir() and _run.parent == RUNS_DIR and (_run / "weights").is_dir()):
        continue
    _dst = PROJECT / "runs" / "detect" / _run.name
    _dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(_run, _dst, dirs_exist_ok=True)
    _synced.append(_run.name)
    _last = _dst / "weights" / "last.pt"
    assert _last.exists(), f"last.pt missing after sync: {_dst}"
    _best = _dst / "weights" / "best.pt"
    _msg = (f"best.pt {_best.stat().st_size/1e6:.1f} MB"
            if _best.exists() and _best.stat().st_size > 40_000_000
            else "last.pt only (partial run — resume it, don't finish)")
    print(f"Synced {_run.name} -> {_dst} | {_msg}")
assert _synced, "No run dirs synced — training produced nothing?"


In [ ]:
!ls -la /content/runs/ 2>/dev/null; du -sh /content/runs/standard_11X_colab*/weights/ 2>/dev/null

In [ ]:
# Manual weights copy: newest standard_* checkpoint -> Drive (auto-finds the run dir).
import shutil
from pathlib import Path
_cands = sorted(
    (p for p in Path("/content/runs").glob("standard_11X_colab*")
     if (p / "weights" / "last.pt").exists()),
    key=lambda p: (p / "weights" / "last.pt").stat().st_mtime, reverse=True)
assert _cands, "No checkpointed selected_* run under /content/runs."
src = _cands[0] / "weights"
dst = Path("/content/drive/MyDrive/Thesis_Workspace/runs/detect") / _cands[0].name / "weights"
dst.mkdir(parents=True, exist_ok=True)
for f in ["best.pt", "last.pt"]:
    if not (src / f).exists():
        print(f"{f} not yet available, skipping")
        continue
    shutil.copy(src / f, dst / f)
    print(f, f"{(dst / f).stat().st_size/1e6:.1f} MB on Drive (from {_cands[0].name})")


In [ ]:
# Resume after a disconnect: run the setup cell (mount) + THIS cell, not the train cell.
# Finds the newest local checkpoint, else pulls the newest one from Drive.
from ultralytics import YOLO
import shutil
from pathlib import Path as _P

RUNS_MIRROR = PROJECT / "runs" / "detect"
_locals = sorted(
    (p for p in RUNS_DIR.glob("standard_11X_colab*")
     if (p / "weights" / "last.pt").exists()),
    key=lambda p: (p / "weights" / "last.pt").stat().st_mtime, reverse=True)
if _locals:
    ckpt = _locals[0] / "weights" / "last.pt"
    print(f"Resuming local: {ckpt}")
else:
    _remote = sorted(
        (p for p in RUNS_MIRROR.glob("standard_11X_colab*")
         if (p / "weights" / "last.pt").exists()),
        key=lambda p: (p / "weights" / "last.pt").stat().st_mtime,
        reverse=True) if RUNS_MIRROR.exists() else []
    assert _remote, "No checkpoint locally or on Drive — run the train cell first."
    ckpt = RUNS_DIR / _remote[0].name / "weights" / "last.pt"
    ckpt.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy(_remote[0] / "weights" / "last.pt", ckpt)
    print(f"Pulled checkpoint from Drive: {_remote[0]}")

model = YOLO(str(ckpt))
model.train(resume=True)

# Sync back the resumed run dir.
_run_dir = _P(getattr(getattr(model, "trainer", None), "save_dir", "") or "")
if not _run_dir or not _run_dir.exists():
    _cands = sorted((p for p in RUNS_DIR.glob("standard_11X_colab*") if p.is_dir()),
                    key=lambda p: p.stat().st_mtime, reverse=True)
    assert _cands, "No run dir found under /content/runs!"
    _run_dir = _cands[0]
assert _run_dir.is_dir() and _run_dir.parent == RUNS_DIR and (_run_dir / "weights").is_dir(), \
    f"Refusing to sync unexpected dir: {_run_dir}"
print(f"Actual run dir: {_run_dir}")

_dst = PROJECT / "runs" / "detect" / _run_dir.name
_dst.parent.mkdir(parents=True, exist_ok=True)
shutil.copytree(_run_dir, _dst, dirs_exist_ok=True)
_best = _dst / "weights" / "best.pt"
assert _best.exists() and _best.stat().st_size > 40_000_000, \
    f"best.pt missing/too small after sync: {_best} (sync copied the wrong dir?)"
print(f"Synced {_run_dir.name} -> {_dst} | best.pt {_best.stat().st_size/1e6:.1f} MB")


In [ ]:
# Finish: publish best.pt to Drive + honest val numbers. Auto-finds the newest
# complete run locally or on Drive, so renamed/bumped run dirs just work.
from ultralytics import YOLO
import shutil
from pathlib import Path as _P

_roots = [d for d in (RUNS_DIR, PROJECT / "runs" / "detect") if d.exists()]
_cands = sorted(
    (p for d in _roots for p in d.glob("standard_11X_colab*")
     if (p / "weights" / "best.pt").exists()
     and (p / "weights" / "best.pt").stat().st_size > 40_000_000),
    key=lambda p: (p / "weights" / "best.pt").stat().st_mtime, reverse=True)
assert _cands, "No complete best.pt anywhere — train/resume first."
best = _cands[0] / "weights" / "best.pt"
out = PROJECT / "models" / "detect" / "best_standard_colab.pt"
out.parent.mkdir(parents=True, exist_ok=True)
shutil.copy(best, out)
assert out.stat().st_size > 40_000_000
print(f"Published {out} ({out.stat().st_size/1e6:.1f} MB) from {_cands[0].name}")

m = YOLO(str(best))
print(f"names: {m.names}")
print(m.val(data=str(DATA_SRC), device=device))  # full val split (plots=True run already saved confusion matrix + PR curves)
import yaml as _yaml
_cfg = _yaml.safe_load(DATA_SRC.read_text(encoding="utf-8"))
if _cfg.get("test"):
    print(m.val(data=str(DATA_SRC), device=device, split="test"))  # held-out test, only if the ndjson carried a test split
else:
    print("No test split in DATA_SRC — val above is the reported metric (add test records to the ndjson for held-out evaluation).")


In [ ]:
# ONE-CLICK RESCUE: copy the newest CHECKPOINTED run to Drive.
# Run this if a finished (or crashed) training never synced. Partial runs
# (last.pt, no best.pt yet) are rescued too.
import shutil
from pathlib import Path as _P

_found = sorted(
    (p for p in RUNS_DIR.glob("standard_11X_colab*")
     if (p / "weights" / "last.pt").exists()),
    key=lambda p: (p / "weights" / "last.pt").stat().st_mtime, reverse=True)
assert _found, "No checkpointed run (weights/last.pt) under /content/runs."
_src = _found[0]
_dst = PROJECT / "runs" / "detect" / _src.name
shutil.copytree(_src, _dst, dirs_exist_ok=True)
_last = _dst / "weights" / "last.pt"
assert _last.exists(), "rescue copy failed!"
_best = _dst / "weights" / "best.pt"
_msg = (f"best.pt {_best.stat().st_size/1e6:.1f} MB"
        if _best.exists() and _best.stat().st_size > 40_000_000
        else "no complete best.pt yet (partial run)")
print(f"Rescued {_src.name} -> {_dst} | last.pt OK | {_msg}")
